# 📬 UC2: Nachrichten-Triage
### Knowledge Navigator 1987 → 2026

**Originalszene:** Phil spielt dem Professor drei Nachrichten vor und gibt einen
priorisierten Überblick — Forschungsteam aus Guatemala, Student Jordan, seine Mutter.

**Heute:** Statt einer Sprachnachricht analysiert eine KI automatisch deine E-Mails
und sortiert sie in Sekunden nach Priorität und Handlungsbedarf.

---
**Lernziele:**
- Claude API aufrufen
- CO-STAR Prompt-Framework anwenden
- Strukturierten JSON-Output verarbeiten
- ipywidgets für interaktive Notebooks nutzen

**Tech Stack:** Python · Anthropic Claude API · ipywidgets · python-dotenv

In [16]:
# ── Setup: Bibliotheken laden & API-Key ─────────────────────────────────────
# Für lokale Entwicklung: API-Key aus .env Datei
# Für Deepnote: Environment Variable im Projekt-Panel setzen
# REGEL: Niemals den API-Key direkt in den Code schreiben!

import json
import os
from pathlib import Path

import anthropic
import ipywidgets as widgets
from dotenv import load_dotenv
from IPython.display import display, HTML

# .env laden (lokal) — auf Deepnote wird die Umgebungsvariable direkt genutzt
load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise ValueError(
        "ANTHROPIC_API_KEY nicht gefunden!\n"
        "Lokal: .env Datei anlegen (siehe .env.example)\n"
        "Deepnote: Environment Variables im Projekt-Panel setzen"
    )

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("✅ Claude-Client initialisiert")

✅ Claude-Client initialisiert


In [17]:
# ── CO-STAR Prompt Template ──────────────────────────────────────────────────
#
# Das CO-STAR-Framework strukturiert LLM-Prompts in 6 Dimensionen:
#
#   C — Context:    Wer ist der Assistent? In welchem Kontext arbeitet er?
#   O — Objective:  Was soll konkret erreicht werden?
#   S — Style:      Wie soll die Antwort formuliert sein?
#   T — Tone:       Welcher Ton ist angemessen?
#   A — Audience:   Für wen ist die Ausgabe bestimmt?
#   R — Response:   Welches Format soll die Antwort haben?
#
# Zusätzlich: Chain-of-Thought — die Kategorien sind explizit definiert,
# damit Claude nachvollziehbar klassifiziert.
# ─────────────────────────────────────────────────────────────────────────────

COSTAR_PROMPT = """\
C (Context): Du bist ein intelligenter E-Mail-Assistent für einen Hochschuldozenten.
Du hilfst dabei, eingehende E-Mails schnell zu priorisieren.

O (Objective): Analysiere die folgende E-Mail. Bestimme Kategorie, Priorität,
erstelle eine Kurzzusammenfassung und empfehle eine konkrete Aktion.

S (Style): Strukturiert, präzise, ohne Füllwörter.

T (Tone): Professionell und sachlich.

A (Audience): Der Dozent möchte in 5 Sekunden entscheiden,
welche Mails sofortige Aufmerksamkeit brauchen.

R (Response): Antworte AUSSCHLIESSLICH mit validem JSON — kein Text davor oder danach:
{{
    \"kategorie\": \"VIP\" | \"Aktion nötig\" | \"Nur Info\" | \"Ignorieren\",
    \"priorität\": 1 | 2 | 3 | 4,
    \"zusammenfassung\": \"Max. 2 prägnante Sätze.\",
    \"empfohlene_aktion\": \"Konkrete, sofort umsetzbare Empfehlung.\"
}}

Kategorien:
- VIP          (Priorität 1): Dekanat, Vorgesetzte, wichtige Partner
- Aktion nötig (Priorität 2): Studierende, Kollegen mit konkreten Anfragen
- Nur Info     (Priorität 3): Newsletter, FYI-Mails, Informationen ohne Handlungsbedarf
- Ignorieren   (Priorität 4): Spam, Werbung, irrelevant

E-Mail:
{email_text}
"""

print("✅ CO-STAR Prompt Template geladen")

✅ CO-STAR Prompt Template geladen


In [18]:
# ── Kernfunktion: analyze_email ──────────────────────────────────────────────

import re

def _strip_code_fences(text: str) -> str:
    """
    Entfernt Markdown-Code-Fences falls Claude JSON in ```json ... ``` verpackt.
    Claude gibt manchmal reines JSON zurück, manchmal mit Code-Fences —
    diese Funktion macht analyze_email robust gegen beide Fälle.
    """
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r'^```(?:json)?\s*', '', text)
        text = re.sub(r'\s*```$', '', text)
    return text.strip()


def analyze_email(email_text: str) -> dict:
    """
    Analysiert eine E-Mail per Claude API und gibt eine strukturierte Triage zurück.

    Args:
        email_text: Vollständiger E-Mail-Text (Betreff + Absender + Body)

    Returns:
        dict: {kategorie, priorität, zusammenfassung, empfohlene_aktion}
    """
    prompt = COSTAR_PROMPT.format(email_text=email_text)

    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )

    raw = _strip_code_fences(response.content[0].text)
    return json.loads(raw)


def format_result_html(result: dict, email_preview: str) -> str:
    """Formatiert das Triage-Ergebnis als farbiges HTML."""
    farben = {
        "VIP":          ("#dc2626", "🔴"),
        "Aktion nötig": ("#d97706", "🟡"),
        "Nur Info":     ("#2563eb", "🔵"),
        "Ignorieren":   ("#6b7280", "⚫"),
    }
    farbe, emoji = farben.get(result["kategorie"], ("#6b7280", "⚫"))
    preview = email_preview[:80].replace('<', '&lt;').replace('>', '&gt;')

    return f"""
    <div style="border-left: 4px solid {farbe}; padding: 12px 16px;
                margin: 8px 0; background: #f9fafb; border-radius: 4px;">
        <div style="font-size: 18px; font-weight: bold; color: {farbe};">
            {emoji} {result['kategorie']} &nbsp;
            <span style="font-size:13px; color:#6b7280;">
                Priorität {result['priorität']}/4
            </span>
        </div>
        <div style="margin-top: 8px; color: #374151;">
            <b>Zusammenfassung:</b> {result['zusammenfassung']}
        </div>
        <div style="margin-top: 4px; color: #374151;">
            <b>Empfehlung:</b> {result['empfohlene_aktion']}
        </div>
        <div style="margin-top: 8px; font-size: 11px; color: #9ca3af;">
            E-Mail-Vorschau: {preview}...
        </div>
    </div>
    """

print("✅ Funktionen definiert")

✅ Funktionen definiert


In [19]:
# ── Interaktives UI mit ipywidgets ───────────────────────────────────────────

email_input = widgets.Textarea(
    placeholder="E-Mail hier einfügen (Von: / Betreff: / Text) ...",
    layout=widgets.Layout(width="100%", height="200px")
)

analyse_btn = widgets.Button(
    description="📬 Analysieren",
    button_style="primary",
    layout=widgets.Layout(width="160px", height="40px")
)

output = widgets.Output()

def on_analyse_click(b):
    with output:
        output.clear_output()
        email_text = email_input.value.strip()
        if not email_text:
            display(HTML("<p style='color:red'>⚠️ Bitte E-Mail-Text einfügen.</p>"))
            return
        display(HTML("<p>⏳ Analysiere...</p>"))
        try:
            result = analyze_email(email_text)
            output.clear_output()
            display(HTML(format_result_html(result, email_text)))
            print("\n📄 Rohes JSON (für Entwickler):")
            print(json.dumps(result, ensure_ascii=False, indent=2))
        except json.JSONDecodeError as e:
            display(HTML(f"<p style='color:red'>❌ JSON-Fehler: {e}</p>"))
        except Exception as e:
            display(HTML(f"<p style='color:red'>❌ Fehler: {e}</p>"))

analyse_btn.on_click(on_analyse_click)

display(
    widgets.HTML("<h3>📬 E-Mail einfügen und analysieren</h3>"),
    email_input,
    analyse_btn,
    output
)

Output()

In [20]:
# ── Demo: Alle 4 Sample-E-Mails auf einmal analysieren ───────────────────────

sample_dir = Path("sample_emails")
sample_files = sorted(sample_dir.glob("*.txt"))

print(f"🔍 Analysiere {len(sample_files)} Sample-E-Mails...\n")

results_html = "<h3>📊 Triage-Ergebnisse</h3>"
for f in sample_files:
    email_text = f.read_text(encoding="utf-8")
    result = analyze_email(email_text)
    results_html += format_result_html(result, email_text)

display(HTML(results_html))

---

## 📡 Stufe 2: Live-Anbindung an Exchange (THWS / DHBW)

Statt E-Mails manuell einzufügen, verbindet sich das Notebook direkt mit dem Exchange-Postfach und analysiert automatisch alle ungelesenen Nachrichten.

**Lernziele Stufe 2:**
- `exchangelib`: E-Mails über EWS (Exchange Web Services) laden
- Sicherer Credential-Dialog mit `ipywidgets`
- **Sicherheitsprinzip:** Credentials nur im RAM — nie auf Disk oder in Git

> ⚠ Zugangsdaten werden **niemals** gespeichert, geloggt oder übertragen.
> Kernel-Neustart löscht alle Credentials automatisch.

In [ ]:
# ── Stufe 2: Exchange-Hilfsfunktionen laden ───────────────────────────────────
# exchange_helpers.py enthält: connect_to_exchange, fetch_emails, build_email_text
# Alle Funktionen sind in tests/test_exchange.py getestet (kein echter Exchange nötig)

from exchange_helpers import (
    INSTITUTIONS,
    build_email_text,
    connect_to_exchange,
    fetch_emails,
)

print("✅ Exchange-Hilfsfunktionen geladen")

In [ ]:
# ── Stufe 2: Credential-Dialog ────────────────────────────────────────────────
#
# Sicherheitsprinzipien (Lehrinhalt — mit Studierenden besprechen):
#   1. Credentials NUR im RAM der Python-Session — nie in Dateien oder Git
#   2. Passwort ist maskiert (widgets.Password → nur Punkte sichtbar)
#   3. Passwort wird nach erfolgreichem Login aus dem Widget gelöscht
#   4. Kernel-Neustart → alle Credentials weg, exchange_account = None
# ─────────────────────────────────────────────────────────────────────────────

exchange_account = None  # Wird nach Verbindung gesetzt

# ── Widgets bauen ─────────────────────────────────────────────────────────────

inst_dropdown = widgets.Dropdown(
    options=list(INSTITUTIONS.keys()),
    value="THWS",
    description="Institution:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="340px"),
)

user_input = widgets.Text(
    placeholder=INSTITUTIONS["THWS"]["username_hint"],
    description="Benutzername:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="340px"),
)

pass_input = widgets.Password(
    placeholder="Passwort",
    description="Passwort:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="340px"),
)

connect_btn = widgets.Button(
    description="🔌 Verbinden",
    button_style="success",
    layout=widgets.Layout(width="150px"),
)

disconnect_btn = widgets.Button(
    description="❌ Trennen",
    button_style="danger",
    layout=widgets.Layout(width="150px"),
    disabled=True,
)

conn_status = widgets.Output()


# ── Platzhalter dynamisch anpassen ────────────────────────────────────────────

def on_inst_change(change):
    inst = change["new"]
    user_input.placeholder = INSTITUTIONS[inst]["username_hint"]

inst_dropdown.observe(on_inst_change, names="value")


# ── Event Handler: Verbinden ──────────────────────────────────────────────────

def on_connect(b):
    global exchange_account
    with conn_status:
        conn_status.clear_output()
        uname = user_input.value.strip()
        pwd = pass_input.value
        inst = inst_dropdown.value

        if not uname or not pwd:
            display(HTML(
                "<p style='color:#dc2626'>⚠️ Benutzername und Passwort erforderlich.</p>"
            ))
            return

        display(HTML("<p>⏳ Verbinde mit Exchange-Server (Autodiscover)...</p>"))
        try:
            exchange_account = connect_to_exchange(uname, pwd, inst)

            total = exchange_account.inbox.total_count
            unread = exchange_account.inbox.unread_count
            smtp = exchange_account.primary_smtp_address

            conn_status.clear_output()
            display(HTML(
                f"<div style='padding:8px; border-left:3px solid #16a34a; background:#f0fdf4'>"
                f"✅ <b>Verbunden:</b> {smtp}<br>"
                f"📬 Posteingang: {total} Mails gesamt, {unread} ungelesen"
                f"</div>"
            ))

            pass_input.value = ""
            connect_btn.disabled = True
            disconnect_btn.disabled = False

        except Exception as e:
            exchange_account = None
            conn_status.clear_output()
            display(HTML(
                f"<div style='padding:8px; border-left:3px solid #dc2626; background:#fef2f2'>"
                f"❌ <b>Verbindungsfehler:</b><br><code>{e}</code><br><br>"
                f"<small>Tipps: Benutzername/Passwort prüfen · "
                f"VPN aktiv? · Exchange-Server erreichbar?</small>"
                f"</div>"
            ))


# ── Event Handler: Trennen ────────────────────────────────────────────────────

def on_disconnect(b):
    global exchange_account
    exchange_account = None
    with conn_status:
        conn_status.clear_output()
        display(HTML(
            "<p style='color:#6b7280'>🔌 Verbindung getrennt — "
            "Credentials aus RAM gelöscht.</p>"
        ))
    connect_btn.disabled = False
    disconnect_btn.disabled = True
    user_input.value = ""


connect_btn.on_click(on_connect)
disconnect_btn.on_click(on_disconnect)


# ── UI rendern ────────────────────────────────────────────────────────────────

display(
    widgets.HTML("<h3>🔐 Exchange-Verbindung</h3>"),
    inst_dropdown,
    user_input,
    pass_input,
    widgets.HTML(
        "<p style='color:#6b7280; font-size:12px; margin:2px 0 8px 0;'>"
        "⚠ Ihre Daten werden nicht gespeichert — "
        "nur im Arbeitsspeicher dieser Jupyter-Session."
        "</p>"
    ),
    widgets.HBox([connect_btn, disconnect_btn]),
    conn_status,
)

In [ ]:
# ── Stufe 2: Live E-Mail-Triage ───────────────────────────────────────────────

max_slider = widgets.IntSlider(
    value=10,
    min=1,
    max=50,
    step=1,
    description="Max. Mails:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="380px"),
)

unread_chk = widgets.Checkbox(
    value=True,
    description="Nur ungelesene E-Mails analysieren",
    indent=False,
)

live_btn = widgets.Button(
    description="📡 Live-Triage starten",
    button_style="primary",
    layout=widgets.Layout(width="210px", height="40px"),
)

live_out = widgets.Output()


def on_live_triage(b):
    with live_out:
        live_out.clear_output()

        if exchange_account is None:
            display(HTML(
                "<p style='color:#dc2626'>"
                "⚠️ Bitte zuerst Exchange verbinden (Zelle oben)."
                "</p>"
            ))
            return

        display(HTML("<p>⏳ Lade E-Mails aus Exchange-Postfach...</p>"))

        try:
            emails = fetch_emails(
                account=exchange_account,
                max_count=max_slider.value,
                unread_only=unread_chk.value,
            )
        except Exception as e:
            live_out.clear_output()
            display(HTML(
                f"<p style='color:#dc2626'>❌ Fehler beim Laden: <code>{e}</code></p>"
            ))
            return

        if not emails:
            live_out.clear_output()
            label = "ungelesene " if unread_chk.value else ""
            display(HTML(
                f"<p style='color:#6b7280'>📭 Keine {label}E-Mails gefunden.</p>"
            ))
            return

        label = "ungelesene " if unread_chk.value else ""
        live_out.clear_output()
        display(HTML(
            f"<p>🤖 Analysiere {len(emails)} {label}E-Mail(s) mit Claude...</p>"
        ))

        results_html = f"<h3>📊 Live-Triage — {len(emails)} E-Mail(s)</h3>"
        errors = 0

        for i, email_dict in enumerate(emails, 1):
            email_text = build_email_text(email_dict)
            try:
                result = analyze_email(email_text)
                results_html += format_result_html(result, email_text)
            except Exception as e:
                errors += 1
                subj = email_dict["subject"][:50]
                results_html += (
                    f"<p style='color:#dc2626'>"
                    f"❌ Fehler bei Mail {i} ({subj}...): {e}"
                    f"</p>"
                )

        live_out.clear_output()

        if errors:
            results_html += (
                f"<p style='color:#d97706'>"
                f"⚠ {errors} Mail(s) konnten nicht analysiert werden.</p>"
            )

        display(HTML(results_html))


live_btn.on_click(on_live_triage)

display(
    widgets.HTML("<h3>📡 Live E-Mail-Triage</h3>"),
    max_slider,
    unread_chk,
    live_btn,
    live_out,
)